In [1]:
pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install sweetviz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, recall_score,  accuracy_score

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# import the data
default = pd.read_csv('/Users/diyapatel/Downloads/creditcard.csv')
default.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


In [5]:
# EDA using sweet viz
report = sv.analyze(default)
report.show_html('sweet_report.html')   


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


For the EDA I wanted to look at Education and it seems that people with different education levels have slightly different credit limits and slightly different ages. But correlations are still moderate at best.

In [6]:
# splitting data into train and test sets
X = default.drop('default payment next month', axis=1)
y = default['default payment next month']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42)      
# training random forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
# evaluating random forest model
y_pred_rf = rf.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
# training xgboost model
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
# evaluating xgboost model
y_pred_xgb = xgb.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.89      3738
           1       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Random Forest Accuracy: 0.81125


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:02:29] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

XGBoost Accuracy: 0.8102083333333333


Both models achieved 81% accuracy, but this is mostly due to predicting the majority class (non-default = 0), where they perform well with a high recall of 93%. However, for the minority class (default = 1), both models performed poorly, with low recall of 38%, meaning they miss a large portion of actual defaults. Overall, Random Forest and XGBoost perform almost identically, and the main issue is class imbalance.

In [7]:
# building baseline models with cross validation 
rf_cv_scores = cross_val_score(rf, X, y, cv=5)
xgb_cv_scores = cross_val_score(xgb, X, y, cv=5)
print("Random Forest CV Scores:", rf_cv_scores)
print("XGBoost CV Scores:", xgb_cv_scores)
print("Random Forest CV Mean Score:", np.mean(rf_cv_scores))
print("XGBoost CV Mean Score:", np.mean(xgb_cv_scores))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:03:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:03:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:03:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:03:40] WARN

Random Forest CV Scores: [0.81125    0.81625    0.814375   0.80958333 0.81875   ]
XGBoost CV Scores: [0.80979167 0.81083333 0.814375   0.81020833 0.81895833]
Random Forest CV Mean Score: 0.8140416666666667
XGBoost CV Mean Score: 0.8128333333333334


In [8]:
rf_cv_recall = cross_val_score(rf, X, y, cv=5, scoring='recall')
xgb_cv_recall = cross_val_score(xgb, X, y, cv=5, scoring='recall')
print("Random Forest CV Recall Scores:", rf_cv_recall)
print("XGBoost CV Recall Scores:", xgb_cv_recall)
print("Random Forest CV Mean Recall Score:", np.mean(rf_cv_recall))
print("XGBoost CV Mean Recall Score:", np.mean(xgb_cv_recall))
print ("Random Forest classification report with recall as evaluation metric:")
print(classification_report(y_test, y_pred_rf))
print ("XGBoost classification report with recall as evaluation metric:")
print(classification_report(y_test, y_pred_xgb))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:05:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:05:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:05:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:05:28] WARN

Random Forest CV Recall Scores: [0.35344015 0.36629002 0.36440678 0.36252354 0.37099812]
XGBoost CV Recall Scores: [0.35626767 0.37288136 0.37193974 0.36817326 0.37382298]
Random Forest CV Mean Recall Score: 0.3635317213090021
XGBoost CV Mean Recall Score: 0.36861699956158334
Random Forest classification report with recall as evaluation metric:
              precision    recall  f1-score   support

           0       0.84      0.93      0.89      3738
           1       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

XGBoost classification report with recall as evaluation metric:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68 

In [9]:
# adjustment to class weights 
rf_weighted = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_weighted.fit(X_train, y_train)
y_pred_rf_weighted = rf_weighted.predict(X_test)
print("Random Forest (Weighted) Classification Report:")
print(classification_report(y_test, y_pred_rf_weighted))
print("Random Forest (Weighted) Accuracy:", accuracy_score(y_test, y_pred_rf_weighted))
xgb_weighted = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', scale_pos_weight= (y == 0).sum() / (y == 1).sum())
xgb_weighted.fit(X_train, y_train)
y_pred_xgb_weighted = xgb_weighted.predict(X_test)
print("XGBoost (Weighted) Classification Report:")
print(classification_report(y_test, y_pred_xgb_weighted))
print("XGBoost (Weighted) Accuracy:", accuracy_score(y_test, y_pred_xgb_weighted))


Random Forest (Weighted) Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.64      0.36      0.46      1062

    accuracy                           0.81      4800
   macro avg       0.74      0.65      0.67      4800
weighted avg       0.79      0.81      0.79      4800

Random Forest (Weighted) Accuracy: 0.8127083333333334


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:06:06] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost (Weighted) Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.84      3738
           1       0.47      0.58      0.52      1062

    accuracy                           0.76      4800
   macro avg       0.67      0.70      0.68      4800
weighted avg       0.79      0.76      0.77      4800

XGBoost (Weighted) Accuracy: 0.7645833333333333


In [10]:
# tuning the random forest 
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10]
}
rf_grid_search = GridSearchCV(estimator=rf, param_grid=rf_param_grid, cv=5, scoring='recall')
rf_grid_search.fit(X_train, y_train)
print("Best Random Forest Hyperparameters:", rf_grid_search.best_params_)
best_rf = rf_grid_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
print("Best Random Forest Classification Report:")
print(classification_report(y_test, y_pred_best_rf))
print("Best Random Forest Accuracy:", accuracy_score(y_test, y_pred_best_rf))


Best Random Forest Hyperparameters: {'max_depth': None, 'n_estimators': 200}
Best Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.63      0.39      0.48      1062

    accuracy                           0.81      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.80      0.81      0.80      4800

Best Random Forest Accuracy: 0.8141666666666667


In [11]:
# adjust for class imbalances 
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6]
}
xgb_grid_search = GridSearchCV(estimator=xgb_weighted, param_grid=xgb_param_grid, cv=5, scoring='recall')
xgb_grid_search.fit(X_train, y_train)
print("Best XGBoost Hyperparameters:", xgb_grid_search.best_params_)
best_xgb = xgb_grid_search.best_estimator_
y_pred_best_xgb = best_xgb.predict(X_test)
print("Best XGBoost Classification Report:")
print(classification_report(y_test, y_pred_best_xgb))
print("Best XGBoost Accuracy:", accuracy_score(y_test, y_pred_best_xgb))    


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:09:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:09:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:09:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:09:59] WARN

Best XGBoost Hyperparameters: {'max_depth': 3, 'n_estimators': 100}
Best XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.79      0.83      3738
           1       0.46      0.64      0.54      1062

    accuracy                           0.75      4800
   macro avg       0.67      0.71      0.68      4800
weighted avg       0.79      0.75      0.77      4800

Best XGBoost Accuracy: 0.75375


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [23:10:06] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


The mean score across folds tells you the model's overall expected performance and how well it performs on average when trained and tested on different subsets of the data. The standard deviation shows how much that performance varies across folds, so it reflects the model's stability. In the model, a low value means consistent performance and a high value means the model is sensitive to the specific data split. 




After tuning the models the performance did increase and this was reflected in the ROC-AUC and Accuracy. Sometimes its necessary to try different parameters to tune to see if the performance is affected by them. 

The Random Forest model with accuracy of .81 was better out of the box as it handled the class imbalances well and performed well on the test dataset without memorizing on the training. It also had a high F1-Score and ROC-AUC. 
